<a href="https://colab.research.google.com/github/busycaesar/Finetune_LoRA/blob/Master/gemma_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import userdata

# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

In [2]:
%pip install -U -q keras-hub keras

In [3]:
os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow".
# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

In [4]:
import keras
import keras_hub
import numpy as np

In [5]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_1b")

100%|██████████| 966/966 [00:00<00:00, 1.45MB/s]


100%|██████████| 3.23k/3.23k [00:00<00:00, 5.12MB/s]


100%|██████████| 4.47M/4.47M [00:00<00:00, 18.4MB/s]


100%|██████████| 1.86G/1.86G [00:30<00:00, 65.8MB/s]


In [9]:
template = "Instruction:\n{instruction}\n\nResponse:\n{response}"

In [10]:
prompt = template.format(
    instruction="I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes.",
    response="",
)
sampler = keras_hub.samplers.TopKSampler(k=5, seed=2)
gemma_lm.compile(sampler=sampler)
print(gemma_lm.generate(prompt, max_length=500))

KeyboardInterrupt: 

In [ ]:
#gemma_lm.generate("I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes.", max_length=500)

In [ ]:
from datasets import load_dataset

ds = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")

# Convert to pandas and sample
df = ds.to_pandas().sample(200, random_state=42)

features = {
    "prompts": df["input"].tolist(),
    "responses": df["output"].tolist()
}

In [ ]:
print(len(features['prompts']))
print(features['prompts'][0])
print(features['responses'][0])

In [ ]:
gemma_lm.backbone.enable_lora(rank=4)

In [ ]:
# Limit the input sequence length to 256 (to control memory usage).
gemma_lm.preprocessor.sequence_length = 256
# Use AdamW (a common optimizer for transformer models).
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)
# Exclude layernorm and bias terms from decay.
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

In [ ]:
gemma_lm.fit(features, epochs=1, batch_size=1)

In [ ]:
prompt = template.format(
    instruction="I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes.",
    response="",
)
sampler = keras_hub.samplers.TopKSampler(k=5, seed=2)
gemma_lm.compile(sampler=sampler)
print(gemma_lm.generate(prompt, max_length=500))

In [ ]:
#gemma_lm.generate("I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes.", max_length=500)

In [ ]:
gemma_lm.save_lora_weights("lora_weights.h5")